# Module 8 — Failure Modes

Part of the RAG Playbook. Program and progress: [CURRICULUM.md](CURRICULUM.md).

## Why you're here: what this module is for

**Where this module starts**
- RAG (retrieval-augmented generation) answers a question in two steps:
  1. A search program finds the few texts in a collection that match the question.
  2. A language model, the kind of program behind chat assistants, reads those texts and writes the answer.
- A RAG system is built from five parts:
  1. Chunk: split each document into short pieces, called chunks
  2. Embed: turn each chunk into an embedding, a list of numbers that an embedding model makes from a text, so that texts with similar meaning get similar embeddings
  3. Store: keep every chunk's embedding in one table
  4. **Retrieve**: for each question, return the chunks that match it best
  5. **Generate**: the language model reads those chunks and writes the answer
- Everything the language model gets, the question included, is called the prompt. In part 5, the prompt holds the returned chunks, each with a number, then the question, then instructions for the answer.

**This module's point:** it builds no new part. It runs the whole system on different kinds of question, finds which kinds it fails on, and traces each failure to the part that caused it: part 4, the search, or part 5, the language model. A failure mode is a kind of question on which the system fails for one shared reason. This module tests four:
1. Questions that need facts from several articles: a company or person described across articles, two articles compared, or two reports compared by date.
2. Questions whose answer is in no article of the collection.
3. Questions whose needed chunk sits in the middle of a long prompt.
4. Questions that ask for a count, such as how many articles mention a company.

<u>Note</u>: the whole RAG system stays the same throughout this module:
- Part 1, chunk: each news article is split into its paragraphs, one chunk per paragraph.
- Part 2, embed: an existing embedding model, nomic-embed-text.
- Part 3, store: has no setting to choose.
- Part 4, retrieve: hybrid search, which combines a meaning search (it compares the question's embedding with each chunk's embedding) and a keyword search (it scores the words a chunk shares with the question). Each chunk gets a hybrid score = 1 / (60 + its rank in the meaning search) + 1 / (60 + its rank in the keyword search), where rank 1 is the best match and 60 is a standard setting. The search returns the chunks with the highest scores while their words stay at or below 400 in total, a number chosen for this playbook, and always at least 1 chunk.
- Part 5, generate: the language model llama3.2:3b, a small, free model that runs on this computer. Its instructions ask for one or two sentences, and set three rules:
  1. Use only the chunks.
  2. After each sentence of the answer, write the numbers of the chunks it comes from in square brackets, such as [2]. This is called citing.
  3. If the chunks do not hold the answer, reply only *"Not enough information."*

**What this module measures**

The test questions come from a public dataset called MultiHop-RAG, of questions about news articles. Each question with an answer was written from two to four articles. For each such question, the dataset gives the correct answer and lists the sentences in those articles that support it, called the listed sentences. This module uses four kinds of question, the dataset's own four:
1. Inference: the correct answer is the name of a company or person that the articles describe.
2. Comparison: the correct answer is yes or no, about whether two or more articles say the same thing.
3. Temporal: the correct answer is yes or no, about whether reports published on different dates agree.
4. Null: no article holds the answer, and the correct answer is *"Insufficient information."*

The four kinds serve the first two failure modes: inference, comparison and temporal questions test failure mode 1, and null questions test failure mode 2. Failure modes 3 and 4 use questions written for this module.

**Two tags in this module**
- **[in real life]** marks a step that happens in a real system: in a test before the system goes live, where the analyst runs test questions that come with a correct answer and reads where the system fails, or with a live user's question.
- **[only in this module]** marks a step that a real system would not do, chosen to make this module simpler or faster.

Each answer is ruled *right* or *not right* by a hand check: a reading of the answer next to the dataset's correct answer. **[in real life]** A person makes the hand check. **[only in this module]** A larger language model made the hand check, by reading each answer, and no person has checked its rulings.

The module runs four steps:
1. Results by kind: the system answers 20 questions of each kind, and the hand check counts the right answers of each kind.
2. Whose fault: the language model answers the same inference, comparison and temporal questions again, from their listed sentences in place of the returned chunks. An answer that turns right shows a search failure, unless the returned chunks already held every listed sentence. An answer that stays not right shows a language model failure.
3. A long prompt: one chunk that holds the answer is placed first, in the middle, or last among about 3,000 words of chunks that do not, to test whether the language model still finds it. In a fourth prompt the chunk is missing, to test whether the language model then declines.
4. Counting: the system answers three questions that ask how many articles mention a company, and each answer is compared with a count made on the article table: the dataset's table of articles, one row per article, with each article's full text.

**What this module produces**
1. For each kind of question, the count of right answers out of 20.
2. For inference, comparison and temporal, the kinds with listed sentences, the not-right answers split into search failures and language model failures.
3. The count of right answers when the chunk that holds the answer sits first, in the middle, or last in a long prompt, or is missing from it.
4. Three counting questions, each with the system's answer next to the count of the article table.

## From the curriculum

> ## <u>Module 8 — Failure Modes</u>
>
> **Concepts**
> - Questions that need several articles, comparisons, and dates
> - Questions with no answer in the collection
> - A relevant chunk lost in a long prompt
> - When RAG is the wrong tool: counting and totals belong in a database query
>
> **Worked example:** the pipeline's results split by question type, with the
> failures traced to their cause.

In this module, the four concepts are the four failure modes listed at the top, in the same order. The worked example is Steps 1 and 2.

The code below loads the tools and sets the answer prompt, the same prompt as the note at the top describes. `common` is this playbook's shared helper file: it loads the data, calls the models, and prints tables.

In [1]:
import re
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

import common
from common import show

BUDGET = 400                                       # words returned per question (part 4)
K1, B, RRF = 1.2, 0.75, 60                         # standard settings, not chosen here: K1 and B of the keyword search (they limit how much repeated words and long chunks raise its score), and RRF, the 60 in the hybrid score
SAMPLE = 20                                        # questions of each kind, chosen for run time
MAX_TOKENS = 120                                   # longest answer, in tokens (word-pieces, about 3/4 of a word each), chosen for run time
KINDS = ["inference", "comparison", "temporal", "null"]
tidy = lambda text: " ".join(text.split())       # one space between words, no line breaks

ANSWER_PROMPT = ("Here are chunks of news articles:\n\n{chunks}\n\nQuestion: {question}\n"
                 "Answer in one or two sentences, using only the chunks above. "
                 "After each sentence of your answer, write the numbers of the chunks it comes from in square brackets, such as [2]. "
                 "If the chunks do not hold the answer, reply only: Not enough information.")
numbered = lambda texts: "\n\n".join(f"[{i}] {t}" for i, t in enumerate(texts, start=1))

## The data: the collection and the test questions

The dataset is MultiHop-RAG (Tang and Yang, 2024):
- 609 news articles, published between September and December 2023, each labelled with one category, such as technology or sports
- 2,556 questions about those articles, of the four kinds listed at the top

**[only in this module]** The collection is the 172 technology articles, not all 609: making the embeddings of all 609 articles' chunks would take hours on this computer.

**[only in this module]** Writing an answer takes the language model about 13 seconds on this computer. So the module uses 20 questions of each kind, a number chosen for run time, drawn at random with a fixed seed, a starting number that makes the same draw come back on every run:
1. Inference, comparison and temporal: only questions whose listed sentences all come from technology articles are drawn. For comparison and temporal, only questions whose correct answer is yes or no, which is almost all of them, as the output below shows.
2. Null: drawn from all null questions. A null question has no answer in any of the 609 articles, so none in the technology articles either.

Steps 1 and 2 then write 140 answers: 4 kinds × 20 = 80 answers from the returned chunks, and 3 kinds × 20 = 60 answers from the listed sentences. 20 questions per kind is small: on another 20 drawn the same way, each count in this module could differ.

In [2]:
corpus = common.load_corpus()
questions = common.load_questions()
articles = corpus[corpus["category"] == "technology"].reset_index(drop=True)
titles = set(articles["title"])
chunks, title_of_chunk = [], []
for title, body in zip(articles["title"], articles["body"]):
    for p in body.split("\n\n"):
        if p.strip():
            chunks.append(tidy(p))
            title_of_chunk.append(title)
words_of_chunk = np.array([len(c.split()) for c in chunks])

in_technology = questions["evidence_list"].map(lambda evidence: bool(evidence) and {e["title"] for e in evidence} <= titles)
yes_or_no = questions["answer"].str.lower().isin(["yes", "no"])
pools = {
    "inference": questions[in_technology & (questions["question_type"] == "inference_query")],
    "comparison": questions[in_technology & (questions["question_type"] == "comparison_query") & yes_or_no],
    "temporal": questions[in_technology & (questions["question_type"] == "temporal_query") & yes_or_no],
    "null": questions[questions["question_type"] == "null_query"],
}
drawn = {kind: pool.sample(SAMPLE, random_state=common.SEED).copy() for kind, pool in pools.items()}

print(f"Technology articles, the collection = {len(articles):,}")
print(f"Chunks, one per paragraph = {len(chunks):,}")
print()
for kind, query_type in [("comparison", "comparison_query"), ("temporal", "temporal_query")]:
    all_of_kind = in_technology & (questions["question_type"] == query_type)
    print(f"{kind.capitalize()} questions from technology articles = {all_of_kind.sum()}, of which the correct answer is yes or no = {(all_of_kind & yes_or_no).sum()}")
print()
print("Questions that can be drawn, by kind:")
print()
show(pd.DataFrame({"kind": KINDS, "questions that can be drawn": [f"{len(pools[k]):,}" for k in KINDS],
                   "drawn": [str(SAMPLE)] * 4}))

Technology articles, the collection = 172
Chunks, one per paragraph = 7,137

Comparison questions from technology articles = 199, of which the correct answer is yes or no = 191
Temporal questions from technology articles = 205, of which the correct answer is yes or no = 194

Questions that can be drawn, by kind:

kind        questions that can be drawn  drawn
inference                           371     20
comparison                          191     20
temporal                            194     20
null                                301     20


**[in real life]** The search runs once per question, and the language model answers from the returned chunks. The code below sets up the search exactly as described in the note at the top, then writes the 80 answers. `common` saves every answer in a file on this computer, so a re-run reads the answers back instead of writing them again.

In [3]:
words_in = lambda text: re.findall(r"[a-z0-9]+", text.lower())


def keyword_search(chunks):
    # BM25 (Best Matching 25), the keyword search's score. K1 and B limit how much repeated words and long chunks raise it. Returns a function: question -> one score per chunk.
    tokens = [words_in(c) for c in chunks]
    n = len(chunks)
    length = np.array([len(t) for t in tokens])
    holding = defaultdict(lambda: ([], []))      # word -> (chunks that hold it, times in each)
    for i, words in enumerate(tokens):
        for w, times in Counter(words).items():
            holding[w][0].append(i)
            holding[w][1].append(times)
    holding = {w: (np.array(rows), np.array(times, dtype=float)) for w, (rows, times) in holding.items()}
    length_part = K1 * (1 - B + B * length / length.mean())

    def scores(question):
        total = np.zeros(n)
        for w in dict.fromkeys(words_in(question)):
            if w in holding:
                rows, times = holding[w]
                rarity_weight = np.log(1 + (n - len(rows) + 0.5) / (len(rows) + 0.5))
                total[rows] += rarity_weight * (times * (K1 + 1)) / (times + length_part[rows])
        return total
    return scores


def ranks_of(scores):
    # Rank 1 = highest score. Ties keep the collection's order.
    order = np.argsort(-scores, kind="stable")
    ranks = np.empty(len(scores), dtype=int)
    ranks[order] = np.arange(1, len(scores) + 1)
    return ranks


unit = lambda m: m / np.linalg.norm(m, axis=1, keepdims=True)
store = unit(common.embed(chunks, prefix="search_document: "))   # nomic-embed-text expects this prefix before a passage, and "search_query: " before a question
keyword = keyword_search(chunks)


def retrieve(question):
    # The positions of the chunks returned for a question, best first: up to 400 words in total, and at least 1 chunk.
    meaning_ranks = ranks_of(store @ unit(common.embed([question], prefix="search_query: "))[0])
    hybrid = 1 / (RRF + meaning_ranks) + 1 / (RRF + ranks_of(keyword(question)))
    order = np.argsort(-hybrid, kind="stable")
    k = max(1, int((np.cumsum(words_of_chunk[order]) <= BUDGET).sum()))
    return list(order[:k])


answer = lambda texts, question: common.generate(ANSWER_PROMPT.format(chunks=numbered(texts), question=question), max_tokens=MAX_TOKENS)
for kind, table in drawn.items():
    table["returned"] = [[chunks[i] for i in retrieve(q)] for q in table["query"]]
    table["answer: returned chunks"] = [answer(r, q) for r, q in zip(table["returned"], table["query"])]

## Step 1 of 4: results by kind

**Which kinds of question does the system get wrong?**

**[in real life]** A person reads every answer and rules it *right* when it gives the dataset's correct answer, and *not right* otherwise:
- On a yes/no question, an answer that declines, gives neither yes nor no, or gives both, is *not right*. When a yes/no answer restates the question as a statement instead of writing *yes* or *no*, the hand check reads whether the statement means *yes* or *no* and rules accordingly.
- On a null question, an answer is *right* only when it declines and gives no answer, because no article holds one.
- An answer that gives an answer and then adds *"Not enough information"* is ruled by the answer it gives.

To keep the code short, the hand check below lists only the answers ruled *not right*, each with its reason. Every answer it does not list is *right*.

In [4]:
# Kind -> question number (1-20 within each kind) -> reason. Every answer not listed is ruled right.
NOT_RIGHT_RETURNED = {
    "inference": {
        5: "the answer names Google, where the correct answer is Apple",
        6: "the answer names Meta, where the correct answer is Google",
        13: "the answer names Google, where the correct answer is Apple",
    },
    "comparison": {
        4: "the answer says The Verge does not claim it: a no, where the correct answer is yes",
        7: "the answer leaves out the Engadget part of the question, so it does not answer the whole question",
        14: "the answer says the two discounts are the same: a yes, where the correct answer is no",
        15: "the answer declines: it says no chunk mentions Starz",
        16: "the answer says the two articles differ: a yes, where the correct answer is no",
        18: "the answer declines: it says the chunks do not show how withdrawal permissions were set",
    },
    "temporal": {
        1: "the answer says TechCrunch failed to keep its reporting consistent: a yes, where the correct answer is no",
        2: "the answer says the rationale is inconsistent: a no, where the correct answer is yes",
        4: "the answer gives neither yes nor no",
        5: "the answer gives both: the same portrayal, then different details",
        6: "the answer gives both: consistent, then not possible to determine",
        7: "the answer says the plan is not mentioned: a no, where the correct answer is yes",
        10: "the answer says the rationale is inconsistent: a yes, where the correct answer is no",
        13: "the answer gives neither yes nor no",
        14: "the answer gives both: consistent, then not consistently portrayed",
        15: "the answer gives neither yes nor no",
        17: "the answer gives neither yes nor no",
        19: "the answer gives neither yes nor no",
        20: "the answer says the portrayal did not change: a yes, where the correct answer is no",
    },
    "null": {
        1: "the answer names the Transport Minister",
        2: "the answer names the prominence measure",
        6: "the answer gives the letter S",
        7: "the answer gives the letter G",
        11: "the answer gives the letters S and C",
        12: "the answer gives the letter N",
        14: "the answer names Brazil",
        15: "the answer names silica",
        16: "the answer names the district of Washington, D.C.",
        17: "the answer names Anker",
        18: "the answer gives the letter V",
        19: "the answer gives the letter T",
    },
}
for kind, table in drawn.items():
    table["hand: returned chunks"] = ["not right" if n in NOT_RIGHT_RETURNED[kind] else "right" for n in range(1, SAMPLE + 1)]

rows = [{"kind": kind, "questions": str(SAMPLE),
         "hand check: right": str((drawn[kind]["hand: returned chunks"] == "right").sum()),
         "hand check: not right": str(len(NOT_RIGHT_RETURNED[kind]))} for kind in KINDS]
rows.append({"kind": "total", "questions": str(SAMPLE * 4),
             "hand check: right": str(sum(int(r["hand check: right"]) for r in rows)),
             "hand check: not right": str(sum(int(r["hand check: not right"]) for r in rows))})
print("Answers from the returned chunks, by kind:")
print()
show(pd.DataFrame(rows))

Answers from the returned chunks, by kind:

kind        questions  hand check: right  hand check: not right
inference          20                 17                      3
comparison         20                 14                      6
temporal           20                  7                     13
null               20                  8                     12
total              80                 46                     34


**Observation:** the system gets most inference questions right, and fails more often on the other three kinds. The temporal questions fail most often: 13 of 20 answers are *not right*.

**Observation:** on the null questions, 12 of 20 answers give an answer, although no article holds one. Rule 3 asks the language model to reply *"Not enough information"* when the chunks lack the answer, and the language model broke rule 3 on those 12. The output below prints one of them, picked by hand as an example, with its returned chunks, to show where its answer came from. Every answer in this module is printed word for word, as the language model wrote it.

In [5]:
row = drawn["null"].iloc[13]
print(f"null question 14: {row['query']}")
print(f"  Answer: {tidy(row['answer: returned chunks'])}")
print()
print("Its returned chunks:")
for i, c in enumerate(row["returned"], start=1):
    print(f"[{i}] {c}")

null question 14: Considering the economic analysis from Bloomberg and the environmental policies discussed in The Guardian, which country in South America, facing criticism for deforestation, is also projected to experience a significant increase in GDP growth in the next fiscal year?
  Answer: Brazil is projected to experience a significant increase in GDP growth in the next fiscal year, facing criticism for deforestation. [1] Not enough information.

Its returned chunks:
[1] Fatoumata Bâ, founder and executive chair of Janngo Capital, affirmed the firm’s support for StarNews, accentuating the platform’s unique positioning, strategic partnerships, technology and content. This support is rooted in the belief that StarNews is well-positioned to capitalize on the immense growth potential of Africa’s creative industry, which is poised for a fourfold expansion by 2050 and the possibility to generate up to $20 billion in GDP, she said.
[2] What would have been a historic deal to tackle a p

**Observation:** the answer names *Brazil*, cites chunk 1, and then adds *"Not enough information."* No returned chunk mentions Brazil or deforestation, so the name did not come from the chunks. It came from the question's wording or from the language model's training text, the text it learned from long before the question. The search cannot cause this failure: a null question has no right chunk to return, so declining is the language model's job alone.

## Step 2 of 4: whose fault

**When an answer is not right, did the search miss the facts, or did the language model misuse them?**

An answer from the returned chunks can fail in two places:
- a search failure: the returned chunks lack facts the question needs
- a language model failure: the facts are there, but the answer is still not right

**[in real life]** In a test before the system goes live, the analyst separates the two by giving the language model the listed sentences in place of the returned chunks, with the same prompt. The listed sentences hold every fact the question needs, by the dataset's design. Each answer then gets one of four labels, from its two rulings:

| listed sentences ↓ / returned chunks → | right | not right |
|---|---|---|
| **right** | right from both | search failure |
| **not right** | unconfirmed right | language model failure |

One rule comes before the table: when the returned chunks already held every listed sentence, a *not right* answer from them is a language model failure, whatever the answer from the listed sentences. The search found every fact, so the language model is the part that failed.

The split is rough, for two reasons:
- The listed sentences also make a much shorter prompt than the returned chunks, so an answer can turn right because the prompt is shorter, not only because it holds the facts.
- On a yes/no question, an answer from the listed sentences can be right by chance, so a few answers counted as search failures may not be.

The null questions have no listed sentences, so this step covers the other three kinds. The code below writes the 60 answers from the listed sentences, and the hand check rules them with the same rules as in Step 1.

In [6]:
NOT_RIGHT_LISTED = {
    "inference": {
        18: "the answer names Google, where the correct answer is Apple",
    },
    "comparison": {
        9: "the answer says the article suggests altruism: a yes, where the correct answer is no",
        12: "the answer affirms the different contexts: a yes, where the correct answer is no",
        13: "the answer says the article implies a reduced ease: a yes, where the correct answer is no",
        14: "the answer says the two discounts are the same: a yes, where the correct answer is no",
        16: "the answer says the two articles differ: a yes, where the correct answer is no",
    },
    "temporal": {
        1: "the answer says the reporting appears inconsistent: a yes, where the correct answer is no",
        2: "the answer says the rationale is inconsistent: a no, where the correct answer is yes",
        4: "the answer gives neither yes nor no",
        5: "the answer says the portrayal appears inconsistent: a yes, where the correct answer is no",
        7: "the answer says one of the three parts is not explicitly mentioned",
        9: "the answer gives neither yes nor no",
        12: "the answer gives neither yes nor no",
        14: "the answer says the second report gives no information: a no, where the correct answer is yes",
        15: "the answer gives neither yes nor no",
        17: "the answer says the reporting might not be entirely negative: a yes, where the correct answer is no",
        19: "the answer gives neither yes nor no",
        20: "the answer gives neither yes nor no",
    },
}
WITH_LISTED = ["inference", "comparison", "temporal"]
for kind in WITH_LISTED:
    table = drawn[kind]
    table["listed"] = table["evidence_list"].map(lambda evidence: [tidy(e["fact"]) for e in evidence])
    table["answer: listed sentences"] = [answer(s, q) for s, q in zip(table["listed"], table["query"])]
    table["hand: listed sentences"] = ["not right" if n in NOT_RIGHT_LISTED[kind] else "right" for n in range(1, SAMPLE + 1)]
    flat = lambda text: " ".join(text.split()).lower()
    table["returned holds every listed sentence"] = [all(flat(s) in flat(" ".join(r)) for s in listed)
                                                     for listed, r in zip(table["listed"], table["returned"])]


def label(row):
    returned, listed = row["hand: returned chunks"], row["hand: listed sentences"]
    if returned == "right":
        return "right from both" if listed == "right" else "unconfirmed right"
    if listed == "right" and not row["returned holds every listed sentence"]:
        return "search failure"
    return "language model failure"


LABELS = ["right from both", "search failure", "unconfirmed right", "language model failure"]
rows = []
for kind in WITH_LISTED:
    drawn[kind]["label"] = drawn[kind].apply(label, axis=1)
    counts = drawn[kind]["label"].value_counts()
    rows.append({"kind": kind, **{name: str(counts.get(name, 0)) for name in LABELS}, "total": str(SAMPLE)})
rows.append({"kind": "total", **{name: str(sum(int(r[name]) for r in rows)) for name in LABELS}, "total": str(SAMPLE * 3)})
total = rows[-1]

print(f"Answers by their rulings, all three kinds ({SAMPLE * 3} answers):")
print()
cell = lambda name: f"{name}: {total[name]}"
print(f"{'listed sentences / returned chunks':<36}{'right':<22}not right")
print(f"{'right':<36}{cell('right from both'):<22}{cell('search failure')}")
print(f"{'not right':<36}{cell('unconfirmed right'):<22}{cell('language model failure')}")
print()
print("Answers by their labels, by kind:")
print()
show(pd.DataFrame(rows))

Answers by their rulings, all three kinds (60 answers):

listed sentences / returned chunks  right                 not right
right                               right from both: 32   search failure: 9
not right                           unconfirmed right: 6  language model failure: 13

Answers by their labels, by kind:

kind        right from both  search failure  unconfirmed right  language model failure  total
inference                16               3                  1                       0     20
comparison               11               4                  3                       2     20
temporal                  5               2                  2                      11     20
total                    32               9                  6                      13     60


**Observation:** the three kinds fail for different reasons:
- Inference: all 3 not-right answers are search failures. Given the listed sentences, the language model names the right company each time.
- Comparison: 4 of the 6 not-right answers are search failures, and 2 are language model failures.
- Temporal: 11 of the 13 not-right answers are language model failures. With every listed sentence in front of it, the language model still fails to say whether two reports agree.

**Observation:** 6 answers are unconfirmed right: *right* from the returned chunks, *not right* from the listed sentences. Such a right ruling does not show that the system worked, for two possible reasons:
- On a yes/no question, the system can land on the correct yes or no by chance.
- The dataset's correct answer can itself be wrong, as the third example below shows.

The output below prints three examples, each with its listed sentences: a search failure, a language model failure, and an unconfirmed right answer.

In [7]:
def show_pair(kind, n):
    row = drawn[kind].iloc[n - 1]
    print(f"{kind} question {n}: {row['query']}")
    print(f"  Correct answer: {row['answer']}")
    print(f"  Answer from the returned chunks: {tidy(row['answer: returned chunks'])}")
    print(f"  Answer from the listed sentences: {tidy(row['answer: listed sentences'])}")
    print("  Listed sentences:")
    for s in row["listed"]:
        print(f"    - {s}")
    print()


print("A search failure:")
show_pair("inference", 5)
print("A language model failure:")
show_pair("comparison", 14)
print("Unconfirmed right:")
show_pair("inference", 18)

A search failure:
inference question 5: Which company, according to articles from TechCrunch and The Verge, not only learns from the missteps of its competitors to deliver superior products but also enforces uniform terms on developers through its store and payment system while simultaneously justifying its choice of a default search engine on its devices due to the lack of a valid alternative?
  Correct answer: Apple
  Answer from the returned chunks: Google is the default search engine on many devices because it is the best in the business, and users prefer it due to its scale, longevity, and ubiquity, despite the lack of a valid alternative. [1] [3] [2]
  Answer from the listed sentences: The company in question is Apple, which learns from competitors' failures and enforces uniform terms on developers, while also choosing a default search engine due to the lack of a valid alternative. [1, 3]
  Listed sentences:
    - Apple has an established pattern of waiting, watching and learning

**Observation:**
- Inference question 5: from the returned chunks, the answer names *Google*. The first listed sentence names Apple, and from the listed sentences the answer names *Apple*.
- Comparison question 14: both answers say the two discounts are the same, *$7 off* (meaning *yes*). The first listed sentence gives $7 off the Jenga game. The second, which the dataset lists for the smart plug, gives 36 percent off. So the language model misread the facts it was given.
- Inference question 18: the listed sentences describe Google, not Apple: Eddy Cue said there was no valid alternative to Google, Google held its hardware event, and Google is accused of harming news publishers. So the answer from the listed sentences, *Google*, matches the listed sentences, and the dataset's correct answer, *Apple*, looks wrong. A test question's correct answer can itself be wrong, and only reading the answers shows it. So the Step 1 count of 17 right inference answers includes this one: it was ruled *right* because it matches the dataset's correct answer, *Apple*, which looks wrong.

**Conclusion:** on these questions, most inference and comparison failures came from the search, and most temporal failures came from the language model. So the fix differs by kind: a better search can help only where the returned chunks lacked the facts.

## Step 3 of 4: a long prompt

**Does the language model still find a needed chunk when it sits in the middle of a long prompt?**

In this system, the prompt holds at most 400 words of chunks, unless the best chunk alone is longer. A system that returns more chunks, to make it more likely that the needed one is among them, builds a longer prompt, and the needed chunk can then sit anywhere in it. A study of several language models (Liu and others, 2023, *Lost in the Middle*) found that they use a fact in the middle of a long prompt less reliably than one near its start or end. This step tests whether llama3.2:3b does so.

**[only in this module]** The test uses ten questions written for this module, not the dataset's. Each asks for one number that one chunk of the collection states, such as the price of an acquisition. The correct number cannot be read off the question's wording, unlike a dataset inference question, which describes its company in such detail that the answer may come from the question's wording instead of the needed chunk. For each question, the prompt holds the needed chunk and about 3,000 words of chunks from other technology articles, picked at random with the fixed seed. The needed chunk is placed:
- first
- in the middle
- last
- nowhere: the prompt then lacks the number, so the right reply is *"Not enough information"*. This tests whether the language model declines when the needed chunk is missing.

Each long prompt takes the language model about 100 seconds on this computer, against about 13 seconds for a prompt of 400 words: a longer prompt also costs time. The answers are ruled by hand, *right* when they give the number the needed chunk states.

In [8]:
FILLER_WORDS = 3000                                # words of chunks that do not hold the answer, chosen for run time
NUM_CTX = 8192                                     # the language model's reading window, in tokens: raised from common's 4,096 to fit the long prompt
NEEDLES = [  # (words that find the needed chunk, the question, the correct answer)
    ("covering 110 percent of the P3 color space with a peak brightness", "What is the peak brightness of the Steam Deck OLED's screen, in nits?", "1,000"),
    ("an active beta program with 150 users", "How many users are in Keep Labs' beta program?", "150"),
    ("just $14.5 billion invested in Chinese companies", "How much did U.S.-headquartered venture capital firms invest in Chinese companies in 2022, according to PitchBook?", "14.5 billion"),
    ("Snap spent more than $200 million to acquire Zenly", "How much did Snap spend to acquire Zenly?", "200 million"),
    ("That $360 million, for instance", "How large was the alleged payment that kept Activision from opening an app store that could compete with Google Play?", "360 million"),
    ("emit 6 million tons of tire wear particles", "How many tons of tire wear particles do cars and trucks emit every year, according to researchers at Imperial College London?", "6 million"),
    ("within 14 days of your purchase", "Within how many days of a purchase will Target match a competitor's price?", "14"),
    ("Dig Security for an estimated $400 million", "For how much is Palo Alto Networks acquiring Dig Security?", "400 million"),
    ("50 percent of US adults say they are unlikely to buy an EV", "According to Pew Research, what percent of US adults say they are unlikely to buy an EV?", "50"),
    ("has picked up €473 million", "How many euros did Cubic Telecom pick up from SoftBank Corp?", "473 million"),
]
POSITIONS = ["first", "middle", "last", "nowhere"]
rng = np.random.default_rng(common.SEED)
long_rows, prompts = [], {}
print("The ten questions, each with its correct answer:")
for n, (_, question, correct) in enumerate(NEEDLES, start=1):
    print(f"  {n}. {question} Correct answer: {correct}")
for n, (key, question, correct) in enumerate(NEEDLES, start=1):
    needed = next(i for i, c in enumerate(chunks) if key in c)
    filler, words = [], 0
    for i in rng.permutation([i for i, t in enumerate(title_of_chunk) if t != title_of_chunk[needed]]):
        if words + words_of_chunk[i] <= FILLER_WORDS:
            filler.append(chunks[i])
            words += words_of_chunk[i]
        if words > FILLER_WORDS - 40:
            break
    mid = len(filler) // 2
    layouts = {"first": [chunks[needed]] + filler, "middle": filler[:mid] + [chunks[needed]] + filler[mid:],
               "last": filler + [chunks[needed]], "nowhere": filler}
    for position in POSITIONS:
        prompts[(n, position)] = layouts[position]
        reply = common.generate(ANSWER_PROMPT.format(chunks=numbered(layouts[position]), question=question),
                                max_tokens=MAX_TOKENS, num_ctx=NUM_CTX)
        long_rows.append({"question": question, "correct": correct, "position": position, "answer": reply})
long_answers = pd.DataFrame(long_rows)

The ten questions, each with its correct answer:
  1. What is the peak brightness of the Steam Deck OLED's screen, in nits? Correct answer: 1,000
  2. How many users are in Keep Labs' beta program? Correct answer: 150
  3. How much did U.S.-headquartered venture capital firms invest in Chinese companies in 2022, according to PitchBook? Correct answer: 14.5 billion
  4. How much did Snap spend to acquire Zenly? Correct answer: 200 million
  5. How large was the alleged payment that kept Activision from opening an app store that could compete with Google Play? Correct answer: 360 million
  6. How many tons of tire wear particles do cars and trucks emit every year, according to researchers at Imperial College London? Correct answer: 6 million
  7. Within how many days of a purchase will Target match a competitor's price? Correct answer: 14
  8. For how much is Palo Alto Networks acquiring Dig Security? Correct answer: 400 million
  9. According to Pew Research, what percent of US adults s

The hand check below lists the long-prompt answers ruled *not right*, by question number (1 to 10, as printed above) and position. Every answer it does not list is *right*. The output prints the counts by position, then every answer ruled *not right*, with a check of whether any chunk in its prompt holds the number it gave: a number held by no chunk was made up by the language model. With the needed chunk nowhere, the correct reply is *"Not enough information"*: an answer that declines is *right*, and one that gives a number is *not right*.

In [9]:
NOT_RIGHT_LONG = {
    # (question number, position): reason
    (3, "nowhere"): "the answer gives $43 billion",
    (4, "nowhere"): "the answer declines, then gives $100 million from outside the chunks",
    (5, "nowhere"): "the answer gives $100 million",
    (6, "nowhere"): "the answer gives 1.5 million tons",
    (7, "nowhere"): "the answer gives 15 days",
    (9, "nowhere"): "the answer gives 44 percent",
}
MADE_UP = {3: r"43 billion", 4: r"100 million", 5: r"100 million", 6: r"1\.5 million", 7: r"15 days", 9: r"44( percent|%)"}   # the number each answer gave
holding_number = lambda n: [i for i, c in enumerate(prompts[(n, "nowhere")], start=1) if re.search(MADE_UP[n], c)]
LEFT_OUT = {8: "one of the other chunks in its four prompts, from another article, also gives Dig Security's price, as between $300 million and $400 million, so an answer may come from that chunk instead of the needed one"}
long_answers["number"] = np.repeat(np.arange(1, len(NEEDLES) + 1), len(POSITIONS))
long_answers["hand"] = ["not right" if (n, p) in NOT_RIGHT_LONG else "right" for n, p in zip(long_answers["number"], long_answers["position"])]
for n, reason in LEFT_OUT.items():
    print(f"Question {n} is left out of the counts, because {reason}.")
    print()
counted = long_answers[~long_answers["number"].isin(list(LEFT_OUT))]
rows = [{"needed chunk": p, "questions": str((counted["position"] == p).sum()),
         "hand check: right": str(((counted["position"] == p) & (counted["hand"] == "right")).sum()),
         "hand check: not right": str(((counted["position"] == p) & (counted["hand"] == "not right")).sum())} for p in POSITIONS]
print("Answers from a long prompt, by where the needed chunk sits:")
print()
show(pd.DataFrame(rows))
print()
for (n, p), reason in NOT_RIGHT_LONG.items():
    row = long_answers[(long_answers["number"] == n) & (long_answers["position"] == p)].iloc[0]
    print(f"question {n}, needed chunk {p}: {row['question']}")
    print(f"  Correct answer: {row['correct']}")
    print(f"  Answer: {tidy(row['answer'])}")
    print(f"  Hand check: not right, because {reason}")
    print(f"  Chunks in the prompt that hold the answer's number: {', '.join(str(i) for i in holding_number(n)) or 'none'}")
    print()
print("Question 5, the two chunks of its prompt that hold the answer's number:")
for i in holding_number(5):
    print(f"[{i}] {prompts[(5, 'nowhere')][i - 1]}")

Question 8 is left out of the counts, because one of the other chunks in its four prompts, from another article, also gives Dig Security's price, as between $300 million and $400 million, so an answer may come from that chunk instead of the needed one.

Answers from a long prompt, by where the needed chunk sits:

needed chunk  questions  hand check: right  hand check: not right
first                 9                  9                      0
middle                9                  9                      0
last                  9                  9                      0
nowhere               9                  3                      6

question 3, needed chunk nowhere: How much did U.S.-headquartered venture capital firms invest in Chinese companies in 2022, according to PitchBook?
  Correct answer: 14.5 billion
  Answer: According to PitchBook, U.S.-headquartered venture capital firms invested around $43 billion in Chinese companies in 2022. [56] Not enough information.
  Hand check

**Observation:** with the needed chunk in the prompt, the language model gave the number the needed chunk states on all 9 questions, whether the chunk sat first, in the middle or last. So at about 3,000 words of other chunks, this test finds no chunk lost in the middle. Longer prompts were not tested, because each would take several minutes on this computer, so a chunk lost in a longer prompt remains possible.

**Observation:** with the needed chunk nowhere, the language model gave a number on 6 of the 9 questions, each different from the number the needed chunk states. For 5 of the 6, no chunk in the prompt holds the number, as the check under each answer shows. For question 5, the answer's *$100 million* matches chunks 56 and 69, which count users of apps, not a payment, as the two chunks printed above show. In all 6, the language model gave a number that no chunk gives for what the question asks, instead of replying *"Not enough information"*, as on the null questions in Step 1.

**Conclusion:** at this prompt length, a needed chunk in the middle was found as reliably as one at the start or end. A missing chunk was the failure: the language model gave a number the chunks do not support.

## Step 4 of 4: counting

**Can the system answer how many articles mention a company?**

**[in real life]** A live user asks a counting question, such as *"How many of the technology articles mention OpenAI?"*. The search returns chunks up to 400 words, and the language model can count only what those chunks show. The correct count needs every article: it is a count over the article table, one line of code with no language model.

The code below asks three counting questions, for OpenAI, Netflix and Nvidia, each name picked by hand as an example. For each, it prints:
- the count of the article table: the technology articles whose text holds the name as a whole word, in the exact spelling shown
- the returned chunks, and how many of them hold the name
- how many different articles the returned chunks that mention it come from
- the system's answer

In [10]:
rows, replies = [], []
for name in ["OpenAI", "Netflix", "Nvidia"]:
    question = f"How many of the technology articles mention {name}?"
    holds = lambda text: bool(re.search(r"\b" + name + r"\b", text))
    returned = retrieve(question)
    holding = [i for i in returned if holds(chunks[i])]
    replies.append((question, answer([chunks[i] for i in returned], question)))
    rows.append({"name": name, "articles that mention it": str(articles["body"].map(holds).sum()),
                 "returned chunks": str(len(returned)), "returned chunks that mention it": str(len(holding)),
                 "articles of the chunks that mention it": str(len({title_of_chunk[i] for i in holding}))})
print("Counting questions: the article table's count next to what the search returned:")
print()
show(pd.DataFrame(rows))
print()
for question, reply in replies:
    print(question)
    print(f"  Answer: {tidy(reply)}")
    print()

Counting questions: the article table's count next to what the search returned:

name     articles that mention it  returned chunks  returned chunks that mention it  articles of the chunks that mention it
OpenAI                         23                6                                6                                       4
Netflix                        13                9                                6                                       3
Nvidia                          7                5                                5                                       5

How many of the technology articles mention OpenAI?
  Answer: Three technology articles mention OpenAI. [1], [2], [4] [1] mentions OpenAI as the company behind ChatGPT and GPT-X LLMs. [2] mentions OpenAI as the company behind ChatGPT and its rivalry with Anthropic. [4] mentions OpenAI as the company behind GPT-X LLMs and its potential impact on its rivals.

How many of the technology articles mention Netflix?
  Answer:

**Observation:** the system answers three to all three questions, below the article table's count each time. The returned chunks come from only a few articles, because the search returns at most 400 words, or one longer chunk, so no answer from them could reach the table's count. The prompt numbers chunks, not articles, so the language model cannot even tell how many articles its own chunks come from.

**Conclusion:** the system cannot answer how many articles mention a company: a count needs every article, so it belongs in a query over the article table, not in RAG.

## Key takeaways
- A failure mode is a kind of question the system fails on for one shared reason, and splitting the results by kind of question is what shows it: one count of right answers over all kinds would hide which kind fails.
- Giving the language model the listed sentences in place of the returned chunks separates the two failures. An answer not right from the returned chunks but right from the listed sentences was a search failure, unless the returned chunks already held every listed sentence. An answer not right from both was a language model failure.
- On these questions, most inference and comparison failures came from the search, and most temporal failures came from the language model. So a better search can help only where the returned chunks lacked the facts.
- On a question with no answer, the search always returns chunks, so only the language model can decline, and this one often did not.
- A counting question needs every article, and the search returns at most 400 words. A count belongs in a query over the article table, not in RAG.
- These counts are rough, for three reasons:
  - each kind has only 20 questions;
  - a language model, not a person, made the hand check;
  - a dataset's correct answer can itself be wrong, as the correct answer of inference question 18 looks to be.

## Conclusion
A RAG system fails differently on different kinds of question, so each failure has to be traced to the search or the language model before choosing a fix, and some questions, such as counts, need a different tool altogether.